In [38]:
import torch
from torch import nn
import numpy as np
from string import Template
import math

import os 
import sys
sys.path.append(os.path.abspath("./"))
from input_image_gen import generate_input_file
from conv_layer_generator import *
from dense_layer_generator import *
from codebooks_defs_generator import *

In [39]:
def conv_out_size(in_size, stride, padding, k_size):
    return int(((in_size - k_size + (2 * padding)) / stride) + 1) 

def max_pool_out_size(in_size, pool_size, stride, padding):
    return int(((in_size - pool_size + (2 * padding)) / (stride)) + 1)

In [40]:

TILE_L2_SIZE = 2
TILE_L1_SIZE = 15

CODEBOOK_SIZE = 4
SVE_LANES = 4

N_LEARNERS = 4

In [41]:
# import torchvision
# from torchsummary import summary

# rs = torchvision.models.resnet18(pretrained=True)

# print(rs)

# summary(rs, (3, 32, 32))

In [42]:

USE_F16 = False

SAME_SEQ = True

USE_BIAS = False

USE_CODEBOOKS = True

OUTPUT_SIZE = 200   # TinyImageNet


IN_CHANNELS = 3
IN_HEIGHT = 64
IN_WIDTH = IN_HEIGHT


####################
##  ResNet18   ##
####################

conv_0 = {"type": "conv",
          "in_ch": IN_CHANNELS, 
          "out_ch": 64,
          "k_size": 3, 
          "stride": 1, 
          "padding": 1}

max_pool_1 = {"type": "maxpool",
              "size": 3,
              "stride": 2,
              "padding": 1}


## ## ## ## ## 
### Block 0 ###

conv_0_0 = {"type": "conv",
              "in_ch": conv_0["out_ch"], 
              "out_ch": 64,
              "k_size": 3, 
              "stride": 1, 
              "padding": 1}

conv_0_1 = {"type": "conv",
              "in_ch": conv_0_0["out_ch"], 
              "out_ch": 64,
              "k_size": 3, 
              "stride": 1, 
              "padding": 1}


## ## ## ## ## 
### Block 1 ###

conv_1_0 = {"type": "conv",
              "in_ch": conv_0_1["out_ch"], 
              "out_ch": 64,
              "k_size": 3, 
              "stride": 1, 
              "padding": 1}

conv_1_1 = {"type": "conv",
              "in_ch": conv_1_0["out_ch"], 
              "out_ch": 64,
              "k_size": 3, 
              "stride": 1, 
              "padding": 1}


## ## ## ## ## 
### Block 2 ###

conv_2_0 = {"type": "conv",
              "in_ch": conv_1_1["out_ch"], 
              "out_ch": 128,
              "k_size": 3, 
              "stride": 2, 
              "padding": 1}

conv_2_1 = {"type": "conv",
              "in_ch": conv_2_0["out_ch"], 
              "out_ch": 128,
              "k_size": 3, 
              "stride": 1, 
              "padding": 1}

conv_2_skip = {"type": "conv",
              "in_ch": conv_1_1["out_ch"], 
              "out_ch": 128,
              "k_size": 1, 
              "stride": 2, 
              "padding": 0}


## ## ## ## ## 
### Block 3 ###

conv_3_0 = {"type": "conv",
              "in_ch": conv_2_1["out_ch"], 
              "out_ch": 128,
              "k_size": 3, 
              "stride": 1, 
              "padding": 1}

conv_3_1 = {"type": "conv",
              "in_ch": conv_3_0["out_ch"], 
              "out_ch": 128,
              "k_size": 3, 
              "stride": 1, 
              "padding": 1}


## ## ## ## ## 
### Block 4 ###

conv_4_0 = {"type": "conv",
              "in_ch": conv_3_1["out_ch"], 
              "out_ch": 256,
              "k_size": 3, 
              "stride": 2, 
              "padding": 1}

conv_4_1 = {"type": "conv",
              "in_ch": conv_4_0["out_ch"], 
              "out_ch": 256,
              "k_size": 3, 
              "stride": 1, 
              "padding": 1}

conv_4_skip = {"type": "conv",
              "in_ch": conv_3_1["out_ch"], 
              "out_ch": 256,
              "k_size": 1, 
              "stride": 2, 
              "padding": 0}


## ## ## ## ## 
### Block 5 ###

conv_5_0 = {"type": "conv",
              "in_ch": conv_4_1["out_ch"], 
              "out_ch": 256,
              "k_size": 3, 
              "stride": 1, 
              "padding": 1}

conv_5_1 = {"type": "conv",
              "in_ch": conv_5_0["out_ch"], 
              "out_ch": 256,
              "k_size": 3, 
              "stride": 1, 
              "padding": 1}


## ## ## ## ## 
### Block 6 ###

conv_6_0 = {"type": "conv",
              "in_ch": conv_5_1["out_ch"], 
              "out_ch": 512,
              "k_size": 3, 
              "stride": 2, 
              "padding": 1}

conv_6_1 = {"type": "conv",
              "in_ch": conv_6_0["out_ch"], 
              "out_ch": 512,
              "k_size": 3, 
              "stride": 1, 
              "padding": 1}

conv_6_skip = {"type": "conv",
              "in_ch": conv_5_1["out_ch"], 
              "out_ch": 512,
              "k_size": 1, 
              "stride": 2, 
              "padding": 0}


## ## ## ## ## 
### Block 7 ###

conv_7_0 = {"type": "conv",
              "in_ch": conv_6_1["out_ch"], 
              "out_ch": 512,
              "k_size": 3, 
              "stride": 1, 
              "padding": 1}

conv_7_1 = {"type": "conv",
              "in_ch": conv_7_0["out_ch"], 
              "out_ch": 512,
              "k_size": 3, 
              "stride": 1, 
              "padding": 1}


## ## ## ## ## 

glob_avg_pool = {"type": "glob_avg_pool"}

dense_8 = {"type": "dense",
           "out_size": OUTPUT_SIZE}


NN_structure = [conv_0, max_pool_1,
                conv_0_0, conv_0_1,
                conv_1_0, conv_1_1,
                conv_2_0, conv_2_1, conv_2_skip,
                conv_3_0, conv_3_1,
                conv_4_0, conv_4_1, conv_4_skip,
                conv_5_0, conv_5_1,
                conv_6_0, conv_6_1, conv_6_skip,
                conv_7_0, conv_7_1,
                glob_avg_pool, dense_8
]

In [43]:
# OUT_FOLDER = "./generated_headers/"
OUT_FOLDER = "./../ResNet18_definitions/"

generate_cb_definitions(OUT_FOLDER + "codebooks_def.h", N_LEARNERS, CODEBOOK_SIZE, SVE_LANES, USE_BIAS, USE_F16, SAME_SEQ, USE_CODEBOOKS)
input_values = generate_input_file(OUT_FOLDER + "input_image.h", IN_CHANNELS, IN_HEIGHT, IN_WIDTH, USE_F16)

# These are in case the layer is a conv or max pool
input_ch = IN_CHANNELS
input_height = IN_HEIGHT
input_width = IN_WIDTH

# This is in case the layer is a dense layer
in_size = IN_CHANNELS * IN_HEIGHT * IN_WIDTH


# This list is to save all the kernel values, so to test with torch
kernels = []

# This list is to save all the dense layer values, so   to test with torch
dense_weights = []

# Final torch network, one per learner
network = [nn.ModuleDict({}) for _ in range(N_LEARNERS)]

for lay_cnt, layer in enumerate(NN_structure):
    print("[{}] {}".format(lay_cnt, layer["type"]))
    print("\t", layer)

    if layer['type'] == "conv":
        in_shape = (layer["in_ch"], input_height, input_width)
        out_channels = layer["out_ch"]
        out_height = conv_out_size(input_height, layer["stride"], layer["padding"], layer["k_size"])
        out_width = conv_out_size(input_width, layer["stride"], layer["padding"], layer["k_size"])
        out_shape = (out_channels, out_height, out_width)

        # Get the codebooks values for the learners and generate the header
        cb_values, k_values, biases_values = generate_kernel_header_file(SAME_SEQ, OUT_FOLDER + "conv_header_{}.h".format(lay_cnt), lay_cnt, N_LEARNERS, CODEBOOK_SIZE, out_channels, layer["in_ch"], layer["k_size"], layer['stride'], layer["padding"], TILE_L2_SIZE, TILE_L1_SIZE, USE_F16, USE_CODEBOOKS)
        kernels.append(k_values)


    
        # Per each learner network, force the kernel values and append the layer to the learner network
        for learner in range(N_LEARNERS):

            new_conv = nn.Conv2d(in_channels=layer['in_ch'],
                                                        out_channels=layer["out_ch"],
                                                        kernel_size=(layer["k_size"], layer["k_size"]),
                                                        stride = layer['stride'],
                                                        padding=layer["padding"],
                                                        bias = USE_BIAS)
            with torch.no_grad():
                new_conv.weight.copy_(torch.tensor(k_values[learner]).view(layer["out_ch"], layer['in_ch'], layer["k_size"], layer["k_size"]))

                if USE_BIAS:
                    new_conv.bias.copy_(torch.tensor(biases_values[learner]))

            network[learner]["conv{}".format(lay_cnt)] = new_conv


        input_ch = out_channels
        input_height = out_height
        input_width = out_width

        in_size = out_channels * out_height * out_width


    elif layer["type"] == "maxpool":
        in_shape = (input_ch, input_height, input_width)
        out_channels = input_ch
        out_height = max_pool_out_size(input_height, layer["size"], layer["stride"], layer["padding"])
        out_width = max_pool_out_size(input_width, layer["size"], layer["stride"], layer["padding"])
        out_shape = (out_channels, out_height, out_width)

        # Per each learner network, append the layer to the network
        for learner in range(N_LEARNERS):
            new_maxpool = nn.MaxPool2d(layer["size"], stride=layer["stride"], padding=layer["padding"])
            # new_maxpool = nn.MaxPool2d(layer["size"])
            network[learner]["maxpool{}".format(lay_cnt)] = new_maxpool

        input_ch = out_channels
        input_height = out_height
        input_width = out_width

        in_size = out_channels * out_height * out_width

    elif layer['type'] == "glob_avg_pool":
        in_shape = (input_ch, input_height, input_width)
        out_channels = input_ch
        out_height = 1
        out_width = 1
        out_shape = (out_channels, out_height, out_width)
        
        for learner in range(N_LEARNERS):
            new_glob_avgPool = nn.AdaptiveAvgPool2d((1, 1))
            network[learner]["glob_avgPool{}".format(lay_cnt)] = new_glob_avgPool
        
        input_ch = out_channels
        input_height = out_height
        input_width = out_width

        in_size = out_channels * out_height * out_width

    elif layer["type"] == "dense":
        print("DENSE: ", in_size)
        in_shape = in_size
        out_size = layer["out_size"]

        dense_values, biases_values = generate_template_dense(SAME_SEQ, OUT_FOLDER + "dense_header_{}.h".format(lay_cnt), lay_cnt, N_LEARNERS, CODEBOOK_SIZE, 200, in_size, out_size, USE_F16, USE_CODEBOOKS)
        dense_weights.append(dense_values)

        for learner in range(N_LEARNERS):

            new_dense = nn.Linear(in_shape, layer["out_size"], bias=USE_BIAS)
            
            with torch.no_grad():
                new_dense.weight.copy_(torch.Tensor(dense_values[learner]).view(out_size, in_shape))

                if USE_BIAS:
                    new_dense.bias.copy_(torch.tensor(biases_values[learner]))

            network[learner]["dense{}".format(lay_cnt)] = new_dense

        out_shape = out_size

        in_size = out_shape
        
    else:
        print("ERROR!")
        exit(1)



    print("In shape:", in_shape)
    print("Out shape:", out_shape)
    print()

[0] conv
	 {'type': 'conv', 'in_ch': 3, 'out_ch': 64, 'k_size': 3, 'stride': 1, 'padding': 1}
In shape: (3, 64, 64)
Out shape: (64, 64, 64)

[1] maxpool
	 {'type': 'maxpool', 'size': 3, 'stride': 2, 'padding': 1}
In shape: (64, 64, 64)
Out shape: (64, 32, 32)

[2] conv
	 {'type': 'conv', 'in_ch': 64, 'out_ch': 64, 'k_size': 3, 'stride': 1, 'padding': 1}
In shape: (64, 32, 32)
Out shape: (64, 32, 32)

[3] conv
	 {'type': 'conv', 'in_ch': 64, 'out_ch': 64, 'k_size': 3, 'stride': 1, 'padding': 1}
In shape: (64, 32, 32)
Out shape: (64, 32, 32)

[4] conv
	 {'type': 'conv', 'in_ch': 64, 'out_ch': 64, 'k_size': 3, 'stride': 1, 'padding': 1}
In shape: (64, 32, 32)
Out shape: (64, 32, 32)

[5] conv
	 {'type': 'conv', 'in_ch': 64, 'out_ch': 64, 'k_size': 3, 'stride': 1, 'padding': 1}
In shape: (64, 32, 32)
Out shape: (64, 32, 32)

[6] conv
	 {'type': 'conv', 'in_ch': 64, 'out_ch': 128, 'k_size': 3, 'stride': 2, 'padding': 1}
In shape: (64, 32, 32)
Out shape: (128, 16, 16)

[7] conv
	 {'type': 'c

In [44]:
network[0]

ModuleDict(
  (conv0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (conv3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (conv4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (conv5): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (conv6): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (conv7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (conv8): Conv2d(64, 128, kernel_size=(1, 1), stride=(2, 2), bias=False)
  (conv9): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (conv10): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (conv11): Conv2

In [ ]:
input = torch.Tensor(input_values).view(IN_CHANNELS, IN_HEIGHT, IN_WIDTH)

print(input.shape)

for ens in range(N_LEARNERS):
    print("\n=============== LEARNER {} ===============\n".format(ens))
    
    x = network[ens]['conv0'](input)
    x = torch.relu(x)
    print(x.shape)

    out_start = network[ens]["maxpool1"](x)

    ######## Block 0 ########
    
    x = network[ens]["conv2"](out_start)
    x = torch.relu(x)
    print(x.shape)

    x = network[ens]["conv3"](x)
    x = x + out_start

    out_0 = torch.relu(x)
    print(x.shape)

    #########################  
    ######## Block 1 ########
    
    x = network[ens]["conv4"](out_0)
    x = torch.relu(x)
    print(x.shape)

    x = network[ens]["conv5"](x)
    x = x + out_0

    out_1 = torch.relu(x)
    print(x.shape)
    
    #########################  
    ######## Block 2 ########
    
    x = network[ens]["conv6"](out_1)
    x = torch.relu(x)
    print(x.shape)

    x = network[ens]["conv7"](x)
    
    out_1 = network[ens]["conv8"](out_1)
    x = x + out_1

    out_2 = torch.relu(x)
    print(x.shape)
    
    #########################   
    ######## Block 3 ########
    
    x = network[ens]["conv9"](out_2)
    x = torch.relu(x)
    print(x.shape)

    x = network[ens]["conv10"](x)
    x = x + out_2

    out_3 = torch.relu(x)
    print(x.shape)
    
    #########################  
    ######## Block 4 ########
    
    x = network[ens]["conv11"](out_3)
    x = torch.relu(x)
    print(x.shape)

    x = network[ens]["conv12"](x)
    
    out_3 = network[ens]["conv13"](out_3)
    x = x + out_3

    out_4 = torch.relu(x)
    print(x.shape)
    
    #########################   
    ######## Block 5 ########
    
    x = network[ens]["conv14"](out_4)
    x = torch.relu(x)
    print(x.shape)

    x = network[ens]["conv15"](x)
    x = x + out_4

    out_5 = torch.relu(x)
    print(x.shape)
    
    #########################  
    ######## Block 6 ########
    
    x = network[ens]["conv16"](out_5)
    x = torch.relu(x)
    print(x.shape)

    x = network[ens]["conv17"](x)
    
    out_5 = network[ens]["conv18"](out_5)
    x = x + out_5

    out_6 = torch.relu(x)
    print(x.shape)
    
    #########################   
    ######## Block 7 ########
    
    x = network[ens]["conv19"](out_6)
    x = torch.relu(x)
    print(x.shape)
    # print(x)
    # break

    x = network[ens]["conv20"](x)
    x = x + out_6

    out_7 = torch.relu(x)
    print(x.shape)

    #########################

    x = network[ens]["glob_avgPool21"](out_7)
    x = x.flatten()
    print(x.shape)

    x = network[ens]["dense22"](x)
    print(x.shape)
    print(x)



torch.Size([3, 64, 64])

=============== LEARNER 0 ===============

torch.Size([64, 64, 64])
torch.Size([64, 32, 32])
torch.Size([64, 32, 32])
torch.Size([64, 32, 32])
torch.Size([64, 32, 32])
torch.Size([128, 16, 16])
torch.Size([128, 16, 16])
torch.Size([128, 16, 16])
torch.Size([128, 16, 16])
torch.Size([256, 8, 8])
torch.Size([256, 8, 8])
torch.Size([256, 8, 8])
torch.Size([256, 8, 8])
torch.Size([512, 4, 4])
torch.Size([512, 4, 4])
tensor([[[1.6553e+37, 3.0704e+37, 3.6291e+37, 2.4435e+37],
         [2.9746e+37, 5.5256e+37, 6.5443e+37, 4.4190e+37],
         [3.4428e+37, 6.3987e+37, 7.5919e+37, 5.1337e+37],
         [2.2677e+37, 4.2190e+37, 5.0201e+37, 3.4051e+37]],

        [[1.6784e+37, 3.1506e+37, 3.7698e+37, 2.5836e+37],
         [3.0275e+37, 5.6559e+37, 6.7438e+37, 4.5968e+37],
         [3.5064e+37, 6.5330e+37, 7.7745e+37, 5.2798e+37],
         [2.3151e+37, 4.2931e+37, 5.0913e+37, 3.4370e+37]],

        [[1.6624e+37, 3.0974e+37, 3.6703e+37, 2.4819e+37],
         [3.0482e+37, 5.